## 1: Cleaning Data

In [3]:
#import all the jazz
import pandas as pd
import re #regular expressions, what pim talked abt that time
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab")

import matplotlib.pyplot as plt
from collections import Counter

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/inqsoncharoen/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:

#def a function to clean up the text and we can use it later
def clean_text(text, additional_patterns=None, remove_stopwords=False):
    if not isinstance(text, str):
        return text  #return non-string entries unchanged

    #remove javascript
    cleaned_text = re.sub(r"<.*?>", "", text)

    #apply additional patterns
    if additional_patterns:
        for pattern in additional_patterns:
            cleaned_text = re.sub(pattern, "", cleaned_text)

    #remove shit
    cleaned_text = re.sub(r"\bnarrator\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bmuseum\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bmoving\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bimage\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\broom\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bliving\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bcandidate\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bapprove\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bthis\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bmessage\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bmale_narrator\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bfemale_narrator\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bal_french\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bfrench_man\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bfrench_woman\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\bcaption\b", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"\btext\b", "", cleaned_text, flags=re.IGNORECASE)

    #convert text to lowercase
    cleaned_text = cleaned_text.lower()

    #uncomment the following lines to remove punctuation
    cleaned_text = re.sub(r"[\W_]+", " ", cleaned_text)

    #tokenize and optionally remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        tokens = word_tokenize(cleaned_text)
        cleaned_text = " ".join(word for word in tokens if word.lower() not in stop_words)

    return cleaned_text

In [ ]:

#load the files from drive (change it to the file path if you download this locally)
input_file_dems = "/Users/inqsoncharoen/Documents/Election_Analysis/transcript_dems.csv"
output_file_dems = "/Users/inqsoncharoen/Documents/Election_Analysis/cleaned_dems.csv"

#read the csv
try:
    data = pd.read_csv(input_file_dems)
except Exception as e: #used chat to figure out this part, of in case of an error
    print({e})
    raise #raise an exception and we can tell it what kind of exception

#this goes back to the other stuff we wanna remove
additional_patterns = [
    #r"\".*?\"",  #remove quoted strings if needed
    r"\n",        #remove newline characters
]

#save the new stuff in a new csv
#lambda functions are SO nerdy lmao. Could"ve used a normal def but you could tell the years spent in STEM jail has come through
data_cleaned = data.copy()
for column in data_cleaned.columns:
    if data_cleaned[column].dtype == object:  #this is for processing only the text columns
        data_cleaned[column] = data_cleaned[column].apply(lambda x: clean_text(
            x, additional_patterns=additional_patterns, remove_stopwords=True
        ))

#save it to a new csv
data_cleaned.to_csv(output_file_dems, index=False)

print(f"Saved: {output_file_dems} for Democrats.")

Saved: /Users/inqsoncharoen/Documents/Election_Analysis/cleaned_dems.csv for Democrats.


In [7]:
input_file_repub = "/Users/inqsoncharoen/Documents/Election_Analysis/transcript_repub.csv"
output_file_repub = "/Users/inqsoncharoen/Documents/Election_Analysis/cleaned_repub.csv"

#read the csv
try:
    data = pd.read_csv(input_file_repub)
except Exception as e: #used chat to figure out this part, of in case of an error
    print({e})
    raise

#this goes back to the other stuff we wanna remove
additional_patterns = [
    #r"\".*?\"",  #remove quoted strings if needed
    r"\n",        #remove newline characters
]

data_cleaned = data.copy()
for column in data_cleaned.columns:
    if data_cleaned[column].dtype == object:  #this is for processing only the text columns
        data_cleaned[column] = data_cleaned[column].apply(lambda x: clean_text(
            x, additional_patterns=additional_patterns, remove_stopwords=True
        ))

data_cleaned.to_csv(output_file_repub, index=False)

print(f"Saved: {output_file_repub} for Republicans.")

Saved: /Users/inqsoncharoen/Documents/Election_Analysis/cleaned_repub.csv for Republicans.


### Merging all the ads into one column
While working on lemmatization, we found out that it was far easier to combine all the ads together -- it was a little easier to loop through the dataframes.

In [10]:
#trying yet another damn solution for this blasted class, gon combine first then loop thru w/ lemmatization maybe more easily?
#dems
df1 = pd.read_csv(output_file_dems)

merged_data = [] #new list to store merged cols

for index, row in df1.iterrows(): #iterate over each row... pandas style
    first_col = str(row[0])  #keep year column
    merged_row = " ".join(str(row[i]) for i in range(1, len(row))) #merge the rest

    merged_data.append([first_col, merged_row]) #append them together

df1_merged = pd.DataFrame(merged_data, columns=["Year", "Ads_Merged"]) #make new df, name cols

df1_merged.to_csv("/Users/inqsoncharoen/Documents/Election_Analysis/combined_dems.csv", index=False)

/var/folders/hb/qtg2k1095t583b4c_cgh3d000000gn/T/ipykernel_1807/3653373158.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_col = str(row[0])  #keep year column
/var/folders/hb/qtg2k1095t583b4c_cgh3d000000gn/T/ipykernel_1807/3653373158.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  merged_row = " ".join(str(row[i]) for i in range(1, len(row))) #merge the rest


In [12]:
#trying yet another damn solution for this blasted class, gon combine first then loop thru w/ lemmatization maybe more easily?
#dems
df2 = pd.read_csv(output_file_repub)

merged_data = [] #new list to store merged cols

for index, row in df1.iterrows(): #iterate over each row... pandas style
    first_col = str(row[0])  #keep year column
    merged_row = " ".join(str(row[i]) for i in range(1, len(row))) #merge the rest

    merged_data.append([first_col, merged_row]) #append them together

df2_merged = pd.DataFrame(merged_data, columns=["Year", "Ads_Merged"]) #make new df, name cols

df2_merged.to_csv("/Users/inqsoncharoen/Documents/Election_Analysis/combined_repub.csv", index=False)

/var/folders/hb/qtg2k1095t583b4c_cgh3d000000gn/T/ipykernel_1807/4165806602.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_col = str(row[0])  #keep year column
/var/folders/hb/qtg2k1095t583b4c_cgh3d000000gn/T/ipykernel_1807/4165806602.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  merged_row = " ".join(str(row[i]) for i in range(1, len(row))) #merge the rest
